# dARK 2.0 - Testing Notebook

This notebook demonstrates the dARK 2.0 system interaction, including Authority registration with **encrypted private keys**, NAAN authorization, and ARK lifecycle management.

## Prerequisites

1. **Start Besu:** `docker-compose up -d`
2. **Deploy Contracts:** `python3 deploy.py`
3. **Install Deps:** `pip install web3 cryptography`

## 1. Setup & Connection

In [ ]:
import os
from dotenv import load_dotenv

# Load unified configuration (try ../../.env then ../.env)
if os.path.exists('../../.env'):
    load_dotenv('../../.env')
elif os.path.exists('../.env'):
    load_dotenv('../.env')
else:
    print('⚠️  Warning: No .env file found in ../../.env or ../.env')

import json
import configparser
import binascii
from web3 import Web3
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

# Connect to Blockchain (load from env with fallback)
RPC_URL = os.getenv('DARK_RPC_URL', 'http://localhost:8545')
w3 = Web3(Web3.HTTPProvider(RPC_URL))

# Deployer Account (Besu Development Account) - Also acts as Admin
ADMIN_PRIVATE_KEY = os.getenv('DARK_ADMIN_PRIVATE_KEY', '0xae6ae8e5ccbfb04590405997ee2d52d2b330726137b875053c36d94e974d162f')
admin_account = w3.eth.account.from_key(ADMIN_PRIVATE_KEY)

print(f"✅ Connected: {w3.is_connected()}")
print(f"👑 Admin Account: {admin_account.address}")
print(f"🔗 RPC URL: {RPC_URL}")

## 2. Load Deployed Contracts

In [ ]:
# Read deployment info
config = configparser.ConfigParser()
config.read('../deployed_contracts.ini')

auth_addr = config['Authority']['address']
auth_abi = json.loads(config['Authority']['abi'])
Authority = w3.eth.contract(address=auth_addr, abi=auth_abi)

dark_addr = config['dARK']['address']
dark_abi = json.loads(config['dARK']['abi'])
dARK = w3.eth.contract(address=dark_addr, abi=dark_abi)

print(f"🛡️ Authority at: {auth_addr}")
print(f"📦 dARK at: {dark_addr}")

## 3. Helper Functions (Crypto & TX)
Definitions for encryption/decryption and transaction sending.

In [ ]:
def derive_key(master_key_hex: str, salt: bytes = b'dark-testing-salt') -> bytes:
    """Derive a 32-byte AES key from the master key"""
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=salt,
        iterations=100000,
    )
    return kdf.derive(bytes.fromhex(master_key_hex.replace('0x', '')))

def encrypt_private_key(private_key_hex: str, master_key_hex: str) -> str:
    """Encrypt a private key using the master key (returns hex)"""
    aes_key = derive_key(master_key_hex)
    aesgcm = AESGCM(aes_key)
    nonce = os.urandom(12)
    data = bytes.fromhex(private_key_hex.replace('0x', ''))
    ciphertext = aesgcm.encrypt(nonce, data, None)
    return '0x' + binascii.hexlify(nonce + ciphertext).decode('utf-8')

def send_tx(from_account, contract_func, gas=300000):
    """Send a transaction"""
    tx = contract_func.build_transaction({
        'from': from_account.address,
        'nonce': w3.eth.get_transaction_count(from_account.address),
        'gas': gas,
        'gasPrice': w3.eth.gas_price
    })
    signed = w3.eth.account.sign_transaction(tx, from_account.key)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
    
    if receipt['status'] == 0:
        raise Exception(f"Transaction Reverted! Gas Used: {receipt['gasUsed']}")
    
    return receipt

print("✅ Core helpers ready.")

## 4. Authority Setup (New Logic)
We will generate a **new wallet** for the Authority, encrypt its private key using the Admin's key (simulating a master key), and register it.

In [ ]:
UUID = "auth-uuid-test-02"


# 1. Create a NEW wallet for this Authority
auth_wallet = w3.eth.account.create()
print(f"🆕 Generated Authority Wallet: {auth_wallet.address}")
print(f"🔑 Private Key: {auth_wallet.key.hex()}")

# 2. Encrypt the private key using Admin's key as Master Key
encrypted_key = encrypt_private_key(auth_wallet.key.hex(), ADMIN_PRIVATE_KEY)
print(f"🔐 Encrypted Key: {encrypted_key[:20]}...")

# 3. Fund the new authority wallet so it can transact later
tx_fund = {
    'to': auth_wallet.address,
    'value': w3.to_wei(1, 'ether'),
    'gas': 21000,
    'gasPrice': w3.eth.gas_price,
    'nonce': w3.eth.get_transaction_count(admin_account.address)
}
signed_fund = w3.eth.account.sign_transaction(tx_fund, admin_account.key)
w3.eth.send_raw_transaction(signed_fund.raw_transaction)
print("💸 Funded Authority wallet with 1 ETH")

# 4. Register Authority (Admin Action)
try:
    # Check if already registered (unlikely for random wallet, but good practice)
    try:
        Authority.functions.get_authority(UUID).call()
        print(f"⚠️ Authority '{UUID}' already registered")
    except:
        print(f"Registering Authority '{UUID}'...")
        receipt = send_tx(admin_account, Authority.functions.register_authority(UUID, auth_wallet.address, encrypted_key))
        print(f"✅ Authority Registered! Gas: {receipt['gasUsed']}")

except Exception as e:
    print(f"❌ Registration Failed: {e}")

## 5. NAAN Authorization
Use the **new Authority wallet** to authorize a NAAN.

In [ ]:
NAAN = "99999"

try:
    if Authority.functions.is_authorized(auth_wallet.address, NAAN).call():
        print(f"NAAN '{NAAN}' is already authorized.")
    else:
        print(f"Authorizing NAAN '{NAAN}' with Authority wallet...")
        # NOTE: Signing with auth_wallet, NOT admin_account
        receipt = send_tx(auth_wallet, Authority.functions.authorize_naan(NAAN))
        print(f"✅ Authorized! Gas: {receipt['gasUsed']}")
except Exception as e:
    print(f"❌ Authorization Failed: {e}")

## 6. ARK Creation
Create a new ARK using the authorized NAAN.

In [ ]:
NAME = "secure-doc-v2"
URL = "https://example.com/secure/v2"
CID = "QmSecureHash456"

try:
    if dARK.functions.ark_exists(NAAN, NAME).call():
        print(f"ARK '{NAAN}/{NAME}' already exists.")
    else:
        print(f"Creating ARK '{NAAN}/{NAME}'...")
        # The creator can be any wallet, but usually the authority needs to approve or own it.
        # For this test, we let the Authority wallet create it.
        receipt = send_tx(auth_wallet, dARK.functions.create_ark(NAAN, NAME, URL, CID), gas=500000)
        print(f"✅ Created! Gas: {receipt['gasUsed']}")
except Exception as e:
    print(f"❌ Creation Failed: {e}")

## 7. Verification: Retrieve Encrypted Key
Admin retrieves the encrypted key from the contract to verify storage.

In [ ]:
try:
    print(f"Retrieving key for '{UUID}'...")
    # Admin only function
    stored_encrypted_key = Authority.functions.get_authority_key(UUID).call({'from': admin_account.address})
    print(f"📦 Stored Key: {stored_encrypted_key[:20]}...")
    
    if stored_encrypted_key == encrypted_key:
        print("✅ SUCCESS: Stored key matches generated key!")
    else:
        print("❌ FAILURE: Keys do not match!")
        
except Exception as e:
    print(f"❌ verification Failed: {e}")